# Colab launcher (thin)

Run the cells top to bottom. This notebook only mounts Drive, gets the repo and
data, builds the cache, runs the tests, and starts `src.train`.

- First pass: leave `SMOKE = True` for a GPU pipeline check.
- Real runs: set `SMOKE = False` and choose `CONFIG`.
- Never commit the cache, run folders, checkpoints, or your token.

Resume: rerunning the training cell after a disconnect automatically continues
from `<RUNS_DIR>/<run name>/last.pt`.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/awdtyo/Polyp-Segmentation.git"
DRIVE_ZIP = "/content/drive/MyDrive/kvasir-seg.zip"
RUNS_DIR = "/content/drive/MyDrive/polyp-runs"
CONFIG = "configs/exp001_convnext_base_baseline.yaml"
SMOKE = True  # set False for a real run
EPOCHS = None  # e.g. 5 for the first real-run probe
BATCH_SIZE = None  # e.g. 2 on a T4 if batch 8 runs out of memory
REPO_DIR = Path("/content/polyp")


## 1. Mount Drive and get the repo

`GH_TOKEN` is read from Colab Secrets and is never printed. Add it with the key
icon in the left sidebar (scope: this notebook). If the repo is public you can
still use a token; it only needs read access.


In [ ]:
import subprocess

from google.colab import drive, userdata

drive.mount("/content/drive")

token = userdata.get("GH_TOKEN")
if not token:
    raise RuntimeError(
        "Missing GH_TOKEN. Add it in Colab Secrets (key icon) and enable it for this notebook."
    )


def _auth_url(url: str, secret: str) -> str:
    """Insert a token into an https clone URL without storing it anywhere."""
    if not url.startswith("https://"):
        raise ValueError("REPO_URL must be an https URL")
    return url.replace("https://", f"https://x-access-token:{secret}@", 1)


def _git(args: list[str]) -> subprocess.CompletedProcess:
    """Run git and raise a token-free message on failure."""
    result = subprocess.run(["git", *args], capture_output=True, text=True)
    if result.returncode != 0:
        message = (result.stderr or result.stdout).replace(token, "***")
        raise RuntimeError(f"git failed: {message}")
    return result


if (REPO_DIR / ".git").is_dir():
    _git(["-C", str(REPO_DIR), "pull", _auth_url(REPO_URL, token)])
else:
    _git(["clone", _auth_url(REPO_URL, token), str(REPO_DIR)])

# Do not leave the token in .git/config.
subprocess.run(
    ["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL],
    check=True,
)
del token
head = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()
print("repo ready at", head)


## 2. Install only missing Python packages

Torch is already installed on Colab and is deliberately not touched.


In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "timm": "timm",
    "albumentations": "albumentations",
    "yaml": "pyyaml",
    "pytest": "pytest",
}
missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print("installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("all required Python packages are already present")


## 3. Unzip the dataset and find `images/` + `masks/`

The folder is discovered by searching for a directory that contains both
`images/` and `masks/`; its path is never hardcoded.


In [ ]:
import shutil
import zipfile

if SMOKE:
    DATA_ROOT = None
    print("SMOKE: synthetic data only; skipping the Drive zip.")
else:
    DATA_DIR = Path("/content/data")
    if DATA_DIR.exists():
        shutil.rmtree(DATA_DIR)
    DATA_DIR.mkdir(parents=True)
    with zipfile.ZipFile(DRIVE_ZIP) as archive:
        archive.extractall(DATA_DIR)

    candidates = [
        path
        for path in DATA_DIR.rglob("*")
        if path.is_dir() and (path / "images").is_dir() and (path / "masks").is_dir()
    ]
    if not candidates:
        raise FileNotFoundError(
            "No folder with both images/ and masks/ inside the zip. "
            "Check the archive layout or DRIVE_ZIP."
        )
    DATA_ROOT = max(candidates, key=lambda path: sum(1 for _ in (path / "images").iterdir()))
    n_images = sum(1 for _ in (DATA_ROOT / "images").iterdir())
    if n_images != 1000:
        print(f"warning: found {n_images} images, expected 1000")
    print("data root:", DATA_ROOT, "images:", n_images)


## 4. Build the npz cache

The cache lives in `/content`, outside git, and is rebuilt at the start of every
session from the zip on Drive. `train.py` records its SHA-256 in `run_info.json`.


In [ ]:
import sys

if SMOKE:
    CACHE_PATH = None
    print("SMOKE: skipping the cache build.")
else:
    CACHE_PATH = Path("/content/cache/kvasir_seg_256.npz")
    CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            sys.executable,
            str(REPO_DIR / "scripts" / "build_cache.py"),
            "--config",
            str(REPO_DIR / CONFIG),
            "--root",
            str(DATA_ROOT),
            "--out",
            str(CACHE_PATH),
        ],
        cwd=REPO_DIR,
        check=True,
    )


## 5. Sanity check

Runs the CPU test suite (no real data, no pretrained weights, no full training).


In [ ]:
import sys

subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO_DIR, check=True)


## 6. Train

The committed config is copied to `/content/config_session.yaml` with absolute
`data.root`, `data.cache_path` and `run.out_dir`, then passed to `src.train`.
This keeps the repository config portable while putting checkpoints on Drive.

`--smoke` is a pipeline test on synthetic data (tiny model, 64 px, 3 steps). Its
numbers are not results.


The command is `python -m src.train` (not `python src/train.py`) so the
`src` package is importable; the launcher is otherwise identical to the CLI.


In [ ]:
from pathlib import Path

import subprocess
import sys

import torch
import yaml

if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Only --smoke is allowed on CPU.")

cfg = yaml.safe_load((REPO_DIR / CONFIG).read_text(encoding="utf-8"))
if not SMOKE:
    cfg.setdefault("data", {})
    cfg["data"]["root"] = str(DATA_ROOT)
    cfg["data"]["cache_path"] = str(CACHE_PATH)
cfg.setdefault("run", {})
cfg["run"]["out_dir"] = RUNS_DIR
if EPOCHS is not None:
    cfg.setdefault("train", {})["epochs"] = EPOCHS
if BATCH_SIZE is not None:
    cfg.setdefault("train", {})["batch_size"] = BATCH_SIZE

Path(RUNS_DIR).mkdir(parents=True, exist_ok=True)
session_config = Path("/content/config_session.yaml")
session_config.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")

cmd = [sys.executable, "-m", "src.train", "--config", str(session_config)]
if SMOKE:
    cmd.append("--smoke")
print("training command:", " ".join(cmd))
print("rerun this cell after a disconnect to resume from last.pt")
subprocess.run(cmd, cwd=REPO_DIR, check=True)


## 7. Read the measured memory and timing

After the first real run, copy peak VRAM, mean epoch time and Dice into the
"Colab results" table in `PROGRESS.md`.


In [ ]:
import json
from pathlib import Path

run_name = cfg["run"]["name"] + ("_smoke" if SMOKE else "")
run_dir = Path(RUNS_DIR) / run_name
print((run_dir / "metrics.csv").read_text(encoding="utf-8"))

info = json.loads((run_dir / "run_info.json").read_text(encoding="utf-8"))
print(
    json.dumps(
        {
            key: info.get(key)
            for key in (
                "gpu_name",
                "params_total",
                "peak_gpu_mem_mb",
                "mean_epoch_time_s",
                "epochs_this_session",
            )
        },
        indent=2,
    )
)
